# LSTM

In this IPython notebook, we demonstrate how Long Short-Term Memory (LSTM) networks work. To keep the explanation clear and unambiguous, we reimplement an LSTM network in NumPy and show that it produces exactly the same results as the LSTM implementation provided by PyTorch.

An alternative explanation—already something of a classic—can be found here: https://colah.github.io/posts/2015-08-Understanding-LSTMs/ (it includes helpful diagrams but does not provide a NumPy implementation).

# Instantiate an LSTM in PyTorch

In order to instantiate an LSTM model in PyTorch we need to specify a small number of hyper parameters: (1) dimensionality of input vectors, (2) dimensionality of hidden vectors, (3) number of layers.

In [16]:
import torch
import torch.nn as nn

# Instantiate an LSTM
input_size = 5      # Dimensionality of each input vector
hidden_size = 3     # Number of hidden units
num_layers = 1      # One LSTM layer

lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

# Accessing Parameters

In [17]:
for name, param in lstm.named_parameters():
    print(f"Name: {name}\tShape: {param.shape}")

Name: weight_ih_l0	Shape: torch.Size([12, 5])
Name: weight_hh_l0	Shape: torch.Size([12, 3])
Name: bias_ih_l0	Shape: torch.Size([12])
Name: bias_hh_l0	Shape: torch.Size([12])


As we can see, the model contains weights and biases for two linear transformations. The first transformation (“ih”) maps a 5-dimensional input vector to a 12-dimensional gates vector. The second transformation (“hh”) maps a 3-dimensional hidden vector to the same 12-dimensional gates vector. The dimensionalities of the input and hidden vectors (5 and 3, respectively) are set at instantiation. The 12-dimensions of the gates vector is derived by multiplying the size of the hidden vector (which is 3) by 4, corresponding to the four gates in the architecture.

The two gates vectors—one from the input vector and one from the hidden vector—are added together. Because of this, having two separate biases is redundant; they can be replaced by a single bias equal to their sum. We can also view the combined operation in another way: concatenate the input and hidden vectors, then apply a single linear transformation to the resulting concatenated vector.

# Running LSTM

## Preparing Input
First, we need to prepare input tensors with the correct shapes. The main input, X, is a sequence of input vectors represented as a 3D tensor. Since we set batch_first=True when creating our LSTM, the first dimension corresponds to the batch size. The second dimension represents time, where the sequence unfolds. The third dimension specifies the size of each input vector.

We also specify the initial values of the hidden and cell vectors. Unlike the input tensor (X), these vectors do not include a time dimension. However, because the model contains multiple layers, the first dimension is used to indexe the layers. The second dimension corresponds to the batch, and the third and final dimension represents the components of the hidden and cell vectors. In this architecture, the hidden and cell vectors have the same length, and this length is the same across all layers.

In [18]:
batch_size = 7
seq_len = 4

x = torch.randn(batch_size, seq_len, input_size)
h0 = torch.randn(num_layers, batch_size, hidden_size)
c0 = torch.randn(num_layers, batch_size, hidden_size)

In [19]:
# Forward through PyTorch
with torch.no_grad():
    y_torch, (hn_torch, cn_torch) = lstm(x, (h0, c0))

## Inspecting Output
Let us now look at what the LSTM generates.

In [6]:
y_torch.shape, hn_torch.shape, cn_torch.shape

(torch.Size([7, 4, 3]), torch.Size([2, 7, 3]), torch.Size([2, 7, 3]))

The first returned tensor (y_torch) contains the sequences of hidden vectors for the last (topmost) layer. It is a 3D tensor with the same dimensions as the input tensor X: batch index, time index, and vector components. Then we have a tuple containing the hidden and cell vectors for all layers, but only at the final time step. Each of these tensors has the following dimensions: layer index, batch index, component index. In other words, we either have all steps but only for the last layer or all the layers but only for the last step. To me, this architecture feels somewhat inelegant. I would prefer a design that provides a single tensor containing the hidden vectors for all layers, batches, and time steps.

In the following cell, we loop over the batch index b. For each batch, we extract the final hidden vector from the last hidden layer, as well as the final element from the sequence of last-layer hidden vectors. We then verify that these two vectors are identical, as expected.

In [8]:
time_ind = -1
layer_ind = -1
for batch_ind in range(7):
    print(y_torch[batch_ind, time_ind, :])
    print(hn_torch[layer_ind, batch_ind, :])
    print()

tensor([-0.2168,  0.1036,  0.0130])
tensor([-0.2168,  0.1036,  0.0130])

tensor([-0.1750,  0.1630,  0.1281])
tensor([-0.1750,  0.1630,  0.1281])

tensor([-0.1544,  0.0550,  0.0724])
tensor([-0.1544,  0.0550,  0.0724])

tensor([-0.1755,  0.2342,  0.1765])
tensor([-0.1755,  0.2342,  0.1765])

tensor([-0.2064,  0.0557, -0.0353])
tensor([-0.2064,  0.0557, -0.0353])

tensor([-0.1554,  0.1960,  0.0446])
tensor([-0.1554,  0.1960,  0.0446])

tensor([-0.1652,  0.1148,  0.1286])
tensor([-0.1652,  0.1148,  0.1286])



In [54]:
sigmoid = lambda x : 0.5 * (1.0 + np.tanh(x/2.0))

class np_lstm:

    def __init__(self, w_ih, w_hh, b_ih, b_hh):
        self.w_ih = w_ih.copy()
        self.w_hh = w_hh.copy()
        self.b_ih = b_ih.copy()
        self.b_hh = b_hh.copy()

    def run(self, X, h0, c0):
        seq_len, input_size = X.shape
        hid_size = len(h0)

        H = np.zeros((seq_len, hid_size))
        C = np.zeros((seq_len, hid_size))
        
        h = h0.copy()
        c = c0.copy()
        
        for t in range(seq_len):
            
            x = X[t]
            G  = (x @ self.w_ih.T + self.b_ih)
            G += (h @ self.w_hh.T + self.b_hh)

            i = G[0*hidden_size : 1*hidden_size]
            f = G[1*hidden_size : 2*hidden_size]
            g = G[2*hidden_size : 3*hidden_size]
            o = G[3*hidden_size : 4*hidden_size]

            i = sigmoid(i)
            f = sigmoid(f)
            g = np.tanh(g)
            o = sigmoid(o)

            c = f * c + i * g
            h = o * np.tanh(c)
        
            C[t, :] = c.copy()
            H[t, :] = h.copy()

        return H, C

In [55]:
y_torch[0]

tensor([[-0.0773, -0.1095, -0.0246],
        [-0.0248, -0.1084,  0.1616],
        [-0.3558, -0.1657, -0.0338],
        [-0.0438, -0.0795, -0.0620]])

In [56]:
w_ih = lstm.state_dict()["weight_ih_l0"].cpu().numpy()
w_hh = lstm.state_dict()["weight_hh_l0"].cpu().numpy()
b_ih = lstm.state_dict()["bias_ih_l0"].cpu().numpy()
b_hh = lstm.state_dict()["bias_hh_l0"].cpu().numpy()

In [57]:
m = np_lstm(w_ih, w_hh, b_ih, b_hh)

In [58]:
h0_inp = h0[0,0,:].cpu().numpy()
c0_inp = c0[0,0,:].cpu().numpy()

In [59]:
X_inp = x[0,:,:].cpu().numpy()

In [60]:
H, C = m.run(X_inp, h0_inp, c0_inp)

In [62]:
H.round(4)

array([[-0.0773, -0.1095, -0.0246],
       [-0.0248, -0.1084,  0.1616],
       [-0.3558, -0.1657, -0.0338],
       [-0.0438, -0.0795, -0.062 ]])

In [45]:
h0_inp

array([-1.213453 , -1.1689838, -0.8923165], dtype=float32)